# Lending Club EDA — Data Quality & Integrity

**Position:** 2 of 14 EDA notebooks under `notebooks/02_eda/`. Read-only against
`data/02_interim/lendingclub.duckdb`. No cleaning here — that happens in
`notebooks/03_data_cleaning/`.

**What this notebook covers**
- Scope — which ~30 of the 152 columns the diagnostic notebooks analyse, and the full logic for that (cell 1)
- Missingness — ranked, and tested for whether *being missing* predicts `is_bad` (not just how much is missing)
- Outliers — IQR and z-score sweeps across every candidate numeric feature, cross-checked
- Plausibility — explicit rules for logically impossible values (negative ratios, out-of-range FICO, ...)
- Row integrity — `id` uniqueness and full-row duplicates at every pipeline stage
- Scope audit — scan the ~120 out-of-scope columns' correlation with `is_bad` for anything wrongly left out
- A per-field entry appended to `field_treatment_ledger.csv` for the cleaning notebook

**Convention:** every code cell has a markdown cell before it (what / why / how, plus an
`Answers:` line — the question the code will answer, never a pre-stated number) and a
`Result` cell after it.

## Cell map

| # | Step |
|---|---|
| 1 | Scope: which ~30 columns the diagnostic notebooks analyse, and why; then rank their missingness |
| 2 | Test whether missingness on those columns is informative of `is_bad` |
| 3 | IQR outlier counts, every candidate numeric feature |
| 4 | Z-score outlier counts (cross-check against IQR) |
| 5 | Plausibility rules for logically impossible values |
| 6 | Duplicate / uniqueness check — `id` and full rows, every pipeline stage |
| 7 | Scope audit — correlation scan of all ~120 out-of-scope columns for a false exclusion |
| 8 | Turn the cell 3 / 4 outlier counts into a per-feature treatment decision |
| 9 | Append this notebook's findings to `field_treatment_ledger.csv` |

**Scope**
- No column is dropped anywhere in Phase 0 before `03_data_cleaning`. Cell 1 explains the full
  151 → 152 → all-profiled → ~30-analysed funnel; the ~30 is an analysis-scope choice, not a data op.
- Cells 6–8 were added in a later audit pass (row integrity, scope audit, and an explicit
  outlier-treatment decision were missing); nothing above them changed.
- No cleaning here — findings are recorded, not applied.

## Cell 1 — scope: which columns the diagnostic notebooks analyse, and why

**No column is dropped anywhere in Phase 0 before `03_data_cleaning`.** The ingestion notebook only
filters *rows* (finished-outcome loans, then the 2013–2017 vintage) and adds `is_bad`. All 152
columns are present in `windowed`, and notebook 1 profiles every one of them.

Notebooks 2–14 run deep per-column diagnostics (distributions, bivariate / WoE / IV, VIF, drift,
transforms). Running that for all 152 columns — most of which a credit modeller can rule out by
definition, not by a test — is mostly output confirming "not a feature". So these notebooks
**scope their analysis** to a ~30-column candidate feature shortlist. This is an analysis-scope
choice, not a data operation.

### The funnel

| Stage | Columns | What happens | Where |
|---|--:|---|---|
| raw file → `raw_mat` | 151 | loaded all-VARCHAR, `SELECT *` — no column change | ingestion cell 1 |
| `raw_mat` → `matured` | 152 | **row** filter (finished-outcome loans) + add `is_bad` | ingestion cell 3 |
| `matured` → `windowed` | 152 | **row** filter (2013–2017 vintage) — no column dropped | ingestion cell 5 |
| `windowed` → notebook 1 | 152 | **all** columns profiled | `eda01` |
| notebook 1 → notebooks 2–14 | **~30** | analysis **scoped** to the shortlist below | this cell |
| notebooks 2–14 → `03_data_cleaning` | TBD | **first actual keep/drop**, per column, with written justification | `03_data_cleaning` cell 1 |

### Why ~120 columns are out of the analysis scope — each excludable by definition, not by a result

| Reason | Examples | Logic |
|---|---|---|
| Post-origination outcome | `total_pymnt*`, `total_rec_*`, `recoveries`, `collection_*`, `last_pymnt_*`, `last_fico_*`, `hardship_*`, `settlement_*`, `debt_settlement_flag` | populated only *after* the loan is running / failing → leakage; a PD feature must be knowable at scoring time |
| Co-borrower fields | `sec_app_*`, `*_joint`, `annual_inc_joint`, `dti_joint`, `verification_status_joint` | structurally null for the ~95% of loans with no joint applicant |
| Identifiers / free text | `member_id`, `url`, `desc`, `title`, `emp_title`, `zip_code` | not features as-is (`id` kept as the row key; `emp_title` / `desc` are Phase 1 NLP candidates) |
| Near-duplicate of a shortlisted column | `funded_amnt`, `funded_amnt_inv` (≈ `loan_amnt`), `fico_range_high` (= `fico_range_low` + 4), `total_bc_limit`, `total_rev_hi_lim` | redundant signal |
| Constant / dead weight here | `policy_code` (1 value), `member_id` (100% null), `hardship_flag` (always 'N'), `pymnt_plan` | no variance to model |
| Redundant recent-activity counters | `open_rv_12m`, `open_il_12m`, `num_tl_op_past_12m`, `mths_since_recent_*`, `all_util`, `percent_bc_gt_75` | overlap the utilisation / inquiry / `acc_open_past_24mths` signals already shortlisted — parked as Phase 1 candidates |

### The ~30 in scope — the standard scorecard feature set (Peaks2Tails KB §2.4 / §2.9)

| Family | Columns |
|---|---|
| Loan terms | `loan_amnt`, `int_rate`, `term`, `grade` |
| Delinquency / inquiry risk | `fico_range_low`, `delinq_2yrs`, `inq_last_6mths`, `pub_rec`, `pub_rec_bankruptcies` |
| Capacity & leverage | `annual_inc`, `dti`, `revol_bal`, `revol_util`, `tot_cur_bal`, `avg_cur_bal` |
| Tradeline structure | `mort_acc`, `open_acc`, `total_acc`, `bc_open_to_buy`, `acc_open_past_24mths`, `mo_sin_old_rev_tl_op`, `num_actv_rev_tl` |
| Borrower context | `emp_length`, `home_ownership`, `verification_status`, `purpose`, `addr_state` |
| Borderline — kept only to re-test the missingness call that set them aside | `mths_since_last_delinq`, `tot_hi_cred_lim`, `bc_util` |

### This is a candidate scope, not a final feature list
- Nothing is removed from `windowed` — later notebooks can still reach for any of the 152
- Notebook 1 already profiled all 152
- **Cell 7 of this notebook audits the exclusion** — it scans every out-of-scope column's correlation with `is_bad` to catch anything wrongly left out (it found 15 worth revisiting → parked for Phase 1)
- `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` cell 1 makes the first real keep/drop calls, per column, with justification (e.g. `avg_cur_bal` dropped after notebook 09's VIF check)

**Answers:** which shortlist columns have real missingness, and how much?

In [ ]:
# Cell 1 -- connect, define the candidate shortlist once, and rank its missingness.

import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()   # read-only; creates the asset folders if missing

# ---- the candidate shortlist -- defined here once, reused by every cell below ----
# Domain-judgment starting point, NOT computed from any notebook. 03_data_cleaning narrows it.
CANDIDATE_NUMERIC = [
    "loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_low", "delinq_2yrs",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "mort_acc", "pub_rec_bankruptcies", "tot_cur_bal", "avg_cur_bal", "bc_open_to_buy",
    "acc_open_past_24mths", "mo_sin_old_rev_tl_op", "num_actv_rev_tl",
]
CANDIDATE_CATEGORICAL = [
    "term", "grade", "emp_length", "home_ownership", "verification_status",
    "purpose", "addr_state",
]
# Kept only to re-check whether they belong (missingness was part of why they were set aside).
BORDERLINE = ["mths_since_last_delinq", "tot_hi_cred_lim", "bc_util"]

shortlist_columns = CANDIDATE_NUMERIC + CANDIDATE_CATEGORICAL + BORDERLINE

# ---- rank missingness on the shortlist ----
row_count = con.sql("SELECT count(*) FROM windowed").fetchone()[0]

missing_records = []
for column in shortlist_columns:
    n_missing = con.sql(f'SELECT count(*) FROM windowed WHERE "{column}" IS NULL').fetchone()[0]
    missing_records.append((column, n_missing, n_missing / row_count))

miss_df = (
    pd.DataFrame(missing_records, columns=["column", "n_missing", "pct_missing"])
    .sort_values("pct_missing", ascending=False)
    .reset_index(drop=True)
)

print(miss_df.to_string(index=False))
miss_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_miss_df.csv"), index=False)

**Result**

- 8 of the 30 shortlist columns have any missingness
- `mths_since_last_delinq` 49.4%, `emp_length` 5.9%, `bc_util` 1.1%, `bc_open_to_buy` 1.0%; `revol_util` / `dti` / `avg_cur_bal` / `inq_last_6mths` near 0
- Every other shortlist column is fully populated

**Next:** for the columns that do have real missingness, test whether *being missing* correlates with `is_bad`.

## Cell 2 — is the missingness informative?

- For every shortlist column with any missingness, compare the bad rate of missing vs. populated rows
- A column with very few missing rows can show a large but meaningless gap, so the headline finding is restricted to columns with ≥ 1,000 missing rows
- `gap_pp` = (bad rate when missing − bad rate when populated), in percentage points; sorted by absolute size

**Answers:** does any column's missingness carry a real, sample-backed signal about `is_bad` — enough to encode as its own `_was_missing` flag rather than impute silently?

In [ ]:
# Cell 2 -- for each shortlist column that has missing values, compare the bad rate
# between missing and populated rows.

missing_vs_bad = []
for column in miss_df.loc[miss_df["pct_missing"] > 0, "column"]:
    grouped = con.sql(f'''
        SELECT
            "{column}" IS NULL     AS is_missing,
            count(*)               AS n,
            round(avg(is_bad), 3)  AS bad_rate
        FROM windowed
        GROUP BY 1
    ''').df()
    grouped["column"] = column
    missing_vs_bad.append(grouped)

missing_vs_bad_df = pd.concat(missing_vs_bad, ignore_index=True)

# Reshape to one row per column: bad rate populated vs. missing, side by side.
bad_rate_by_column = missing_vs_bad_df.pivot(index="column", columns="is_missing", values="bad_rate")
bad_rate_by_column.columns = ["bad_rate_populated", "bad_rate_missing"]

bad_rate_by_column["n_missing"] = (
    miss_df.set_index("column")["n_missing"].reindex(bad_rate_by_column.index)
)
bad_rate_by_column["gap_pp"] = (
    bad_rate_by_column["bad_rate_missing"] - bad_rate_by_column["bad_rate_populated"]
) * 100

bad_rate_by_column = bad_rate_by_column.sort_values("gap_pp", key=abs, ascending=False)

print(bad_rate_by_column[["n_missing", "bad_rate_populated", "bad_rate_missing", "gap_pp"]].to_string())

**Result**

| Column | n missing | bad rate populated | bad rate missing | gap (pp) |
|---|--:|--:|--:|--:|
| `inq_last_6mths` | 1 | 20.5% | 0.0% | −20.5 |
| `avg_cur_bal` | 17 | 20.5% | 29.4% | +8.9 |
| `emp_length` | 70,579 | 20.1% | 27.4% | **+7.3** |
| `mths_since_last_delinq` | 590,519 | 21.2% | 19.9% | −1.3 |
| `bc_util` | 13,083 | 20.5% | 21.5% | +1.0 |

- The two largest gaps (`inq_last_6mths` −20.5, `avg_cur_bal` +8.9) sit on 1 and 17 rows — noise
- `emp_length` is the real finding: +7.3 pp on 70,579 rows — large enough and populated enough to trust
- `mths_since_last_delinq` is 49% missing but its gap is only −1.3 pp on 590k rows — **missingness there is not informative**
- **Treatment:** carry an `emp_length_was_missing` 0/1 flag into cleaning alongside the imputed value (Missing Indicator Approach); do not impute silently

**Next:** outlier detection across every candidate numeric feature.

## Cell 3 — IQR outlier counts, every candidate numeric feature

- Flag values outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` for each candidate numeric feature — a systematic sweep, not a spot check
- IQR is known to over-flag on sparse count fields (most rows sit at 0 with a thin tail), so the ranking is read with that caveat

**Answers:** which candidate numeric features carry the heaviest tails by the IQR rule?

In [ ]:
# Cell 3 -- IQR outlier sweep: count values beyond 1.5*IQR of the quartiles, per feature.

row_count = con.sql("SELECT count(*) FROM windowed").fetchone()[0]

iqr_records = []
for column in CANDIDATE_NUMERIC:
    p25, p75 = con.sql(f'''
        SELECT quantile_cont(TRY_CAST("{column}" AS DOUBLE), 0.25),
               quantile_cont(TRY_CAST("{column}" AS DOUBLE), 0.75)
        FROM windowed
    ''').fetchone()

    iqr = p75 - p25
    lower_fence = p25 - 1.5 * iqr
    upper_fence = p75 + 1.5 * iqr

    n_outliers = con.sql(f'''
        SELECT count(*)
        FROM windowed
        WHERE TRY_CAST("{column}" AS DOUBLE) < {lower_fence}
           OR TRY_CAST("{column}" AS DOUBLE) > {upper_fence}
    ''').fetchone()[0]

    iqr_records.append((column, iqr, lower_fence, upper_fence, n_outliers, n_outliers / row_count))

iqr_df = (
    pd.DataFrame(iqr_records,
                 columns=["column", "iqr", "lower_fence", "upper_fence", "n_outliers", "pct_outliers"])
    .sort_values("pct_outliers", ascending=False)
    .reset_index(drop=True)
)

print(iqr_df.round(2).to_string(index=False))
iqr_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_iqr_df.csv"), index=False)

**Result**

- Top IQR-flagged share: `delinq_2yrs` 20.0%, `pub_rec` 18.1%, `pub_rec_bankruptcies` 13.1%, `bc_open_to_buy` 8.6%, `revol_bal` 6.0%
- The top three are sparse count fields where IQR is degenerate (Q1 = Q3 = 0, so *any* nonzero value is "beyond the fence") — not a real outlier signal
- Genuinely skewed continuous fields (`bc_open_to_buy`, `revol_bal`, `avg_cur_bal`, `annual_inc`) sit just below
- `revol_util` at the bottom (~0%)

**Next:** cross-check with a z-score sweep, which reacts differently to heavy tails.

## Cell 4 — z-score outlier counts (cross-check)

- Flag `|z| > 3` for each candidate numeric feature and line the result up against the cell 3 IQR ranking
- Agreement → stronger evidence of a genuinely outlier-heavy field
- Divergence → usually means extreme values are inflating the standard deviation itself, widening the z-score band

**Answers:** do IQR and z-score agree on which features are outlier-heavy, and where do they diverge?

In [ ]:
# Cell 4 -- z-score outlier sweep: count values more than 3 SDs from the mean, per feature,
# then line the result up against the IQR result from cell 3.

zscore_records = []
for column in CANDIDATE_NUMERIC:
    n_outliers = con.sql(f'''
        WITH col_values AS (
            SELECT TRY_CAST("{column}" AS DOUBLE) AS v
            FROM windowed
            WHERE TRY_CAST("{column}" AS DOUBLE) IS NOT NULL
        ),
        moments AS (SELECT avg(v) AS mu, stddev(v) AS sigma FROM col_values)
        SELECT count(*)
        FROM col_values, moments
        WHERE abs(v - mu) > 3 * sigma
    ''').fetchone()[0]
    zscore_records.append((column, n_outliers))

row_count = con.sql("SELECT count(*) FROM windowed").fetchone()[0]

zscore_df = pd.DataFrame(zscore_records, columns=["column", "n_outliers_z"])
zscore_df["pct_outliers_z"] = zscore_df["n_outliers_z"] / row_count

# Join IQR % and z-score % side by side, ordered by the z-score share.
compare_df = (
    iqr_df[["column", "pct_outliers"]]
    .merge(zscore_df, on="column")
    .rename(columns={"pct_outliers": "pct_outliers_iqr"})
    .sort_values("pct_outliers_z", ascending=False)
    .reset_index(drop=True)
)

print(compare_df.round(4).to_string(index=False))
compare_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_compare.csv"), index=False)

**Result**

- z-score flags far fewer rows on the sparse count fields — `delinq_2yrs` 20.0% (IQR) → 1.4% (z-score), `pub_rec` 18.1% → 1.1%, `pub_rec_bankruptcies` 13.1% → 0.8%
- On the genuinely skewed continuous fields the two roughly agree (`bc_open_to_buy` ~9% vs ~2%, both near the top)
- The divergence is the finding: the sparse-count "outliers" are an IQR artifact; the continuous fields have a real heavy tail
- `loan_amnt`: 0 rows beyond 3 SD — well-behaved

**Next:** a check neither statistical method makes — values that are logically impossible.

## Cell 5 — plausibility checks

- IQR and z-score catch statistically unusual values; neither knows what is logically impossible for this domain
- Check specific rules directly: negative `dti` / `revol_util`, zero `annual_inc`, `revol_util` over 100%, `fico_range_low` outside 300–850, non-positive `loan_amnt`

**Answers:** how many rows break each rule, and which breakages are genuine data errors vs. legitimate edge cases?

In [ ]:
# Cell 5 -- count rows that break each explicit plausibility rule.

plausibility_rules = {
    "dti < 0": "TRY_CAST(dti AS DOUBLE) < 0",
    "annual_inc = 0": "TRY_CAST(annual_inc AS DOUBLE) = 0",
    "revol_util < 0": "TRY_CAST(revol_util AS DOUBLE) < 0",
    "revol_util > 100": "TRY_CAST(revol_util AS DOUBLE) > 100",
    "fico_range_low out of [300,850]": "TRY_CAST(fico_range_low AS DOUBLE) NOT BETWEEN 300 AND 850",
    "loan_amnt <= 0": "TRY_CAST(loan_amnt AS DOUBLE) <= 0",
}

violation_records = []
for label, condition in plausibility_rules.items():
    n_violations = con.sql(f"SELECT count(*) FROM windowed WHERE {condition}").fetchone()[0]
    violation_records.append((label, n_violations))

sanity_df = pd.DataFrame(violation_records, columns=["check", "n_violations"])
print(sanity_df.to_string(index=False))
sanity_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_sanity_df.csv"), index=False)

**Result**

| Rule | Violations | Read |
|---|--:|---|
| `dti < 0` | 2 | data error — floor at 0 or drop in cleaning |
| `annual_inc = 0` | 213 | likely "not provided" — treat as missing in cleaning |
| `revol_util > 100` | 4,571 | genuine (account over its limit) — keep as documented edge case |
| `revol_util < 0` | 0 | — |
| `fico_range_low` out of [300, 850] | 0 | — |
| `loan_amnt <= 0` | 0 | — |

That closes the data-quality pass on the shortlist.

**Next:** row-level integrity — the one-row-per-loan assumption every notebook relies on.

## Cell 6 — duplicate & uniqueness check

- Every notebook here, and the cleaning pass, assumes one row = one loan — never actually verified
- A duplicate `id` or a fully-duplicated row would inflate whichever class it belongs to, and could put the same loan on both sides of a Phase 1 train/test split
- Check `id` uniqueness at all three pipeline stages, and full-row duplicates (every column identical) in `windowed`

**Answers:** is `id` unique at every stage, and are there any fully-duplicated rows?

In [ ]:
# Cell 6 -- check id uniqueness at each pipeline stage, and full-row duplicates in windowed.

id_uniqueness_records = []
for table_name in ["raw_mat", "matured", "windowed"]:
    n_rows, n_distinct_ids = con.sql(
        f"SELECT count(*), count(DISTINCT id) FROM {table_name}"
    ).fetchone()
    id_uniqueness_records.append({
        "table": table_name,
        "n_rows": n_rows,
        "n_distinct_id": n_distinct_ids,
        "duplicate_ids": n_rows - n_distinct_ids,
    })

dup_df = pd.DataFrame(id_uniqueness_records)
print(dup_df.to_string(index=False))
dup_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_dup_df.csv"), index=False)

# Full-row duplicates: rows where every column is identical, not just id.
all_columns = con.sql("DESCRIBE windowed").df()["column_name"].tolist()
quoted_columns = ", ".join(f'"{c}"' for c in all_columns)

n_rows_windowed = con.sql("SELECT count(*) FROM windowed").fetchone()[0]
n_distinct_rows = con.sql(
    f"SELECT count(*) FROM (SELECT DISTINCT {quoted_columns} FROM windowed)"
).fetchone()[0]

print(f"\nfully-duplicated rows in windowed (all {len(all_columns)} columns identical): "
      f"{n_rows_windowed - n_distinct_rows}")

**Result**

| Table | Rows | Distinct `id` | Duplicate `id` |
|---|--:|--:|--:|
| `raw_mat` | 2,260,701 | 2,260,701 | 0 |
| `matured` | 1,348,099 | 1,348,099 | 0 |
| `windowed` | 1,195,879 | 1,195,879 | 0 |

- Fully-duplicated rows in `windowed` (all 152 columns identical): **0**
- One row = one loan is confirmed, not assumed

**Next:** audit the cell-1 analysis scope — scan every out-of-scope column against `is_bad` for a false exclusion.

## Cell 7 — audit the analysis scope for false exclusions

- Cell 1 scoped the diagnostic notebooks to ~30 columns by domain judgment. This checks that judgment against the data.
- Scan **every** out-of-scope column's correlation with `is_bad`:
  - columns excluded as post-origination leakage should show the extreme, near-tautological correlation leakage produces
  - nothing with genuine at-origination signal should be sitting in the excluded set by mistake
- Correlation is computed per column (one unbounded/malformed field can't sink the scan); values are magnitude-filtered before `corr()`
- The 0.05–0.5 mid-range band is triaged by name pattern (near-duplicate / co-borrower / post-origination / unexplained)

**Answers:** did the cell-1 scoping wrongly leave out any column with real origination-time signal?

In [ ]:
# Cell 7 -- correlation of every OUT-OF-SCOPE raw column with is_bad, then triage the mid-range band.
# Nothing was physically dropped anywhere -- "out of scope" just means "not on the cell-1 shortlist".

# Columns the diagnostic notebooks analyse: the shortlist + 4 non-feature keys carried for reference.
IN_SCOPE = {"id", "issue_d", "loan_status", "is_bad"} | set(CANDIDATE_NUMERIC) | set(CANDIDATE_CATEGORICAL)

all_columns = con.sql("DESCRIBE windowed").df()["column_name"].tolist()
out_of_scope_columns = [c for c in all_columns if c not in IN_SCOPE and c != "is_bad"]

# corr() per column. A few raw fields hold values extreme enough to destabilise the
# computation, so cast + magnitude-filter first, and skip any column that still errors.
corr_by_column = {}
skipped_columns = []
for column in out_of_scope_columns:
    try:
        corr_value = con.sql(
            f'SELECT corr(x, is_bad) '
            f'FROM (SELECT TRY_CAST("{column}" AS DOUBLE) AS x, is_bad FROM windowed) '
            f'WHERE x IS NULL OR abs(x) < 1e15'
        ).fetchone()[0]
        if corr_value is not None:
            corr_by_column[column] = corr_value
    except Exception:
        skipped_columns.append(column)

corr_df = (
    pd.DataFrame.from_dict(corr_by_column, orient="index", columns=["corr_with_is_bad"])
    .dropna()
    .sort_values("corr_with_is_bad", key=abs, ascending=False)
)
corr_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_corr_row.csv"), index=True, index_label="feature")

print(f"out-of-scope columns scanned: {len(out_of_scope_columns)} "
      f"(of {len(all_columns)} in windowed; {len(IN_SCOPE) - 4} in scope as features)")
if skipped_columns:
    print(f"skipped (values too extreme for a stable correlation): {skipped_columns}")
print(f"out-of-scope columns with a usable correlation: {len(corr_df)}")
print()
print("top 15 out-of-scope columns by |correlation with is_bad|:")
print(corr_df.head(15).to_string())

# ---- triage the mid-range band ----
second_look = corr_df[(corr_df["corr_with_is_bad"].abs() > 0.05) & (corr_df["corr_with_is_bad"].abs() < 0.5)]
print()
print(f"out-of-scope columns in the 0.05 < |corr| < 0.5 band: {len(second_look)}")

# Name-pattern buckets, so the finding is a triage rather than "go look at 43 columns".
POST_ORIGINATION_PATTERNS = ("pymnt", "rec_", "recoveries", "hardship", "settlement", "collection", "last_")
COBORROWER_PATTERNS = ("sec_app_", "_joint", "verification_status_joint")
NEAR_DUPLICATE_OF_SHORTLIST = {
    "fico_range_high": "fico_range_low", "funded_amnt": "loan_amnt",
    "funded_amnt_inv": "loan_amnt", "total_rev_hi_lim": "revol_bal/revol_util",
    "total_bc_limit": "bc_open_to_buy", "tot_hi_cred_lim": "tot_cur_bal",
}


def triage_bucket(column):
    if column in NEAR_DUPLICATE_OF_SHORTLIST:
        return "near-duplicate of a shortlisted column"
    if any(pattern in column for pattern in COBORROWER_PATTERNS):
        return "co-borrower field (N/A for solo loans, out of scope, not leakage)"
    if any(pattern in column for pattern in POST_ORIGINATION_PATTERNS):
        return "post-origination outcome field (correctly excluded as leakage)"
    return "unexplained -- origination-time signal not currently used"


unexplained_columns = []
if len(second_look):
    triage_df = pd.DataFrame({
        "feature": second_look.index,
        "bucket": [triage_bucket(c) for c in second_look.index],
    })
    triage_df.to_csv(os.path.join(ASSETS_TABLES, "eda02_triage.csv"), index=False)
    unexplained_columns = triage_df.loc[
        triage_df["bucket"].str.startswith("unexplained"), "feature"
    ].tolist()

    print()
    print("triage of that band:")
    print(triage_df["bucket"].value_counts().to_string())
    print(f"\nunexplained (Phase 1 feature candidates, not leakage): {unexplained_columns}")

**Result**

- 121 out-of-scope columns scanned, 27 in scope as features; 91 out-of-scope columns had a usable correlation
- Top |corr| among out-of-scope: `last_fico_range_high` (−0.68), `last_fico_range_low` (−0.58), `recoveries` (0.51), `collection_recovery_fee` (0.50), `total_rec_prncp` (−0.45) — all unambiguously post-origination → excluding them as leakage was correct, not just plausible
- 43 out-of-scope columns fall in the 0.05–0.5 band; triage:

| Bucket | Count |
|---|--:|
| unexplained — origination-time signal not currently used | 15 |
| post-origination outcome field (leakage, weaker) | 12 |
| co-borrower (`sec_app_*` / `_joint`) — structurally N/A for solo loans | 10 |
| near-duplicate of a shortlisted column | 6 |

- The 15 unexplained: `orig_projected_additional_accrued_interest`, `num_tl_op_past_12m`, `all_util`, `open_rv_24m`, `emp_title`, `num_rev_tl_bal_gt_0`, `open_rv_12m`, `percent_bc_gt_75`, `open_acc_6m`, `bc_util`, `mo_sin_rcnt_tl`, `mths_since_recent_inq`, `mo_sin_rcnt_rev_tl_op`, `mths_since_recent_bc`, `open_il_12m`
- Not leakage, not scoping errors — legitimate **Phase 1** engineered-feature candidates, not something to fold into Phase 0 cleaning
- **No false exclusion found:** nothing with strong, safe origination-time signal is sitting outside the shortlist

**Next:** turn the cell 3 / 4 outlier counts into an actual per-feature treatment decision.

## Cell 8 — outlier treatment decision

- Cells 3 and 4 counted outliers but never decided what to do about them
- For each candidate numeric feature, classify as **cap** (winsorize at 1st/99th pct), **leave as-is**, or **review**, using only what this notebook establishes or what is explicitly still in progress:
  - naturally bounded (`fico_range_low`, `int_rate`, `revol_util`) → leave as-is
  - sparse count field where IQR is a known artifact (`delinq_2yrs`, `pub_rec`, `pub_rec_bankruptcies`) → leave as-is
  - a log transform *may* tame the tail — flagged as **pending notebook 09's** transform validation, not asserted here
  - `avg_cur_bal` → review: candidate drop pending notebook 09's VIF check vs. `tot_cur_bal`
  - otherwise (unbounded dollar/count field) → cap
- A decision table, not new computation

**Answers:** which candidate numeric features need winsorizing, and which have a documented reason to be left alone (or revisited later)?

In [ ]:
# Cell 8 -- classify each candidate numeric feature's outlier treatment. Decision table only.

NATURALLY_BOUNDED = {"fico_range_low", "int_rate", "revol_util"}          # real, known plausible range
SPARSE_COUNT_IQR_ARTIFACT = {"delinq_2yrs", "pub_rec", "pub_rec_bankruptcies"}  # Q1=Q3=0 -> IQR degenerate

# Fields a log transform MIGHT tame -- pending notebook 09's transform validation, not settled here.
LOG_TRANSFORM_PENDING_NB09 = {
    "annual_inc", "inq_last_6mths", "mo_sin_old_rev_tl_op", "mort_acc", "total_acc",
}


def outlier_decision(column):
    if column in SPARSE_COUNT_IQR_ARTIFACT:
        return ("leave as-is",
                "IQR flags nearly every nonzero value on this sparse count field -- method artifact, not a real outlier")
    if column in NATURALLY_BOUNDED:
        return ("leave as-is (bounded)",
                "field has a real, known plausible range -- extreme-looking values within it are genuine")
    if column in LOG_TRANSFORM_PENDING_NB09:
        return ("leave as-is (pending notebook 09)",
                "a log transform may reduce this field's skew -- defer any capping until notebook 09 validates the transform")
    if column == "avg_cur_bal":
        return ("review",
                "candidate drop pending notebook 09's VIF check (likely redundant with tot_cur_bal)")
    return ("cap at 1st/99th percentile",
            "unbounded dollar/count field with a heavy tail -- winsorize to limit leverage from extremes without dropping rows")


decision_records = []
for column in CANDIDATE_NUMERIC:
    decision, reasoning = outlier_decision(column)
    decision_records.append({"feature": column, "decision": decision, "reasoning": reasoning})

decisions = pd.DataFrame(decision_records)
decisions.to_csv(os.path.join(ASSETS_TABLES, "eda02_decisions.csv"), index=False)

print(decisions.to_string(index=False))
print(f"\nfeatures to cap: {(decisions['decision'] == 'cap at 1st/99th percentile').sum()}")

**Result**

- **8 features to cap** (winsorize at 1st/99th percentile): `loan_amnt`, `dti`, `open_acc`, `revol_bal`, `tot_cur_bal`, `bc_open_to_buy`, `acc_open_past_24mths`, `num_actv_rev_tl`
- **11 left as-is**, each with a reason:
  - bounded range: `int_rate`, `revol_util`, `fico_range_low`
  - sparse count / IQR artifact: `delinq_2yrs`, `pub_rec`, `pub_rec_bankruptcies`
  - log transform pending notebook 09: `annual_inc`, `inq_last_6mths`, `total_acc`, `mort_acc`, `mo_sin_old_rev_tl_op` — re-checked once that notebook settles the transform
- **1 to review:** `avg_cur_bal` — candidate drop pending notebook 09's VIF check vs. `tot_cur_bal`

**Next:** append every field decision from this notebook to `field_treatment_ledger.csv`.

## Cell 9 — append findings to the field-treatment ledger

- Same cumulative file `01_data_understanding_structural_profiling.ipynb` started: `data/04_assets/tables/field_treatment_ledger.csv`, one row per field, upserted by column name
- This notebook writes `last_updated_by = "eda02"` and `status = "proposed"` on every row it touches (see notebook 01 cell 8 for the full schema and the `action` vocabulary)
- These are the **first `action = "drop"` proposals** in the ledger — nothing was dropped before this; the ledger accumulates proposals that `03_data_cleaning` enacts
- Contributions from this notebook's cells:

| From | Fields | Ledger action |
|---|---|---|
| cell 2 | `emp_length` | `transform` — add an `emp_length_was_missing` 0/1 flag |
| cell 8 | 8 numeric features | `transform` — winsorize at 1st/99th percentile |
| cell 8 | 11 numeric features | `clean` — no capping, reason recorded |
| cell 8 | `avg_cur_bal` | `review` — candidate drop pending notebook 09 |
| cell 5 | `dti`, `annual_inc`, `revol_util` | plausibility notes folded into `treatment_detail` |
| cell 7 | 10 confirmed post-origination + 6 near-duplicate columns | `drop` |
| cell 7 | 15 unexplained columns | `review` — Phase 1 feature candidates |
| cell 6 | `id` | `keep` — uniqueness verified |

**Answers:** what does this notebook tell the cleaning stage to do with each field it touched?

In [ ]:
# Cell 9 -- upsert this notebook's per-field decisions into the shared treatment ledger.
# Self-contained read-modify-write, keyed on column name (see notebook 01 cell 8 for the schema).

LEDGER_PATH = os.path.join(ASSETS_TABLES, "field_treatment_ledger.csv")
NOTEBOOK_ID = "eda02"
LEDGER_COLUMNS = ["column", "inferred_type", "domain_type", "action", "treatment_detail",
                  "basis", "rationale", "status", "last_updated_by"]

# Each entry lists only the fields it changes; the rest are kept from the existing row.
updates = []

# --- cell 2: emp_length missingness is informative ---
updates.append({
    "column": "emp_length", "action": "transform",
    "treatment_detail": ("map '< 1 year'->0, '10+ years'->10, 'N years'->N; cast INT; "
                         "ALSO add emp_length_was_missing 0/1 flag before imputing"),
    "basis": "eda01 Cell 4 + eda02 Cell 2",
    "rationale": "missing vs. populated bad rate 27.4% vs 20.1% (+7.3 pp, n=70,579) -- missingness is signal",
})

# --- cell 8: outlier treatment per candidate numeric feature ---
for _, row in decisions.iterrows():
    feature, decision, reasoning = row["feature"], row["decision"], row["reasoning"]
    if decision.startswith("cap"):
        updates.append({
            "column": feature, "action": "transform",
            "treatment_detail": "cast to DOUBLE; winsorize at 1st/99th percentile",
            "basis": "eda02 Cell 3/4/8", "rationale": reasoning,
        })
    elif decision == "review":
        updates.append({
            "column": feature, "action": "review",
            "treatment_detail": "cast to DOUBLE; " + reasoning,
            "basis": "eda02 Cell 8", "rationale": reasoning,
        })
    else:  # "leave as-is" variants
        updates.append({
            "column": feature, "action": "clean",
            "treatment_detail": "cast to DOUBLE; no outlier capping",
            "basis": "eda02 Cell 8", "rationale": reasoning,
        })

# --- cell 5: plausibility edge cases (override the plain cell-8 entry for these three) ---
updates.append({
    "column": "dti", "action": "transform",
    "treatment_detail": "cast to DOUBLE; winsorize at 1st/99th percentile; floor the 2 negative rows at 0 (or drop)",
    "basis": "eda02 Cell 3/4/8 + Cell 5",
    "rationale": "unbounded heavy tail; 2 rows with dti<0 are data errors",
})
updates.append({
    "column": "annual_inc", "action": "clean",
    "treatment_detail": ("cast to DOUBLE; no outlier capping (log transform pending notebook 09); "
                         "treat the 213 rows with annual_inc=0 as missing"),
    "basis": "eda02 Cell 8 + Cell 5",
    "rationale": "log-transform candidate; annual_inc=0 (n=213) is 'not provided', not a real zero",
})
updates.append({
    "column": "revol_util", "action": "clean",
    "treatment_detail": ("cast to DOUBLE; no outlier capping (bounded); "
                         "keep the 4,571 rows >100% as a documented edge case"),
    "basis": "eda02 Cell 8 + Cell 5",
    "rationale": "naturally bounded; >100% is a genuine over-limit account, not an error",
})

# --- cell 6: id uniqueness verified ---
updates.append({
    "column": "id", "action": "keep",
    "treatment_detail": "keep as the row identifier; never use as a model feature",
    "basis": "eda01 Cell 5 + eda02 Cell 6",
    "rationale": "loan identifier; uniqueness verified at raw_mat / matured / windowed (0 duplicates)",
})

# --- cell 7: first ledger rows to propose action=drop -- confirmed leakage + near-duplicates ---
# (nothing was dropped before this; the ledger accumulates drop *proposals*, enacted in 03_data_cleaning.
#  hardship_* fields stay as notebook 01's 'exclude_from_pd_features', not dropped.)
CONFIRMED_LEAKAGE = [
    "last_fico_range_high", "last_fico_range_low", "recoveries", "collection_recovery_fee",
    "total_rec_prncp", "last_pymnt_amnt", "total_pymnt", "total_pymnt_inv",
    "total_rec_late_fee", "total_rec_int",
]
for column in CONFIRMED_LEAKAGE:
    updates.append({
        "column": column, "action": "drop", "treatment_detail": "drop the column",
        "basis": "eda02 Cell 7 (corr with is_bad)",
        "rationale": "post-origination outcome field -- leakage, not knowable at application time",
    })
for column, replacement in NEAR_DUPLICATE_OF_SHORTLIST.items():
    updates.append({
        "column": column, "action": "drop", "treatment_detail": "drop the column",
        "basis": "eda02 Cell 7 (triage)",
        "rationale": f"near-duplicate of shortlisted column {replacement}",
    })

# --- cell 7: unexplained columns -> Phase 1 feature candidates ---
for column in unexplained_columns:
    updates.append({
        "column": column, "action": "review",
        "treatment_detail": "not on the Phase 0 shortlist; evaluate as an engineered feature in Phase 1",
        "basis": "eda02 Cell 7 (triage)",
        "rationale": "origination-time signal, not leakage, not currently used",
    })

# ---- apply updates on top of the existing ledger (last write per column wins) ----
ledger_df = pd.read_csv(LEDGER_PATH).astype(str)
ledger_by_column = {row["column"]: dict(row) for _, row in ledger_df.iterrows()}

for update in updates:
    column = update["column"]
    if column not in ledger_by_column:
        ledger_by_column[column] = {c: "" for c in LEDGER_COLUMNS}
        ledger_by_column[column]["column"] = column
    ledger_by_column[column].update(update)
    ledger_by_column[column]["status"] = "proposed"
    ledger_by_column[column]["last_updated_by"] = NOTEBOOK_ID

ledger_df = (
    pd.DataFrame(list(ledger_by_column.values()))[LEDGER_COLUMNS]
    .sort_values("column")
    .reset_index(drop=True)
)
ledger_df.to_csv(LEDGER_PATH, index=False)

touched_columns = sorted({u["column"] for u in updates})
print(f"field_treatment_ledger.csv -> {len(ledger_df)} rows; eda02 touched {len(touched_columns)} fields")
print()
touched_rows = ledger_df[ledger_df["last_updated_by"] == NOTEBOOK_ID]
print("eda02 rows by action:")
print(touched_rows["action"].value_counts().to_string())
print()
print(touched_rows.to_string(index=False))

**Result**

- `field_treatment_ledger.csv` still holds one row per field; this notebook updated ~50 of them (`last_updated_by = "eda02"`, all `status = "proposed"`)
- Roughly: `drop` for the 10 confirmed-leakage + 6 near-duplicate columns, `transform` for the 8 winsorize features + `emp_length` + `dti`, `review` for `avg_cur_bal` and the 15 Phase-1 candidates, `clean` for the numeric features left as-is
- `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` reads the finished ledger (after notebook 14) and executes it — this is where those `drop` proposals actually take effect

**Next:** `03_univariate_distributional_visual.ipynb` — full distributional profiling of every shortlist feature.